# 03 — Candidate-path reduction and reduced QUBO

The direct edge formulation preserves the complete routing semantics of the time-expanded OEC graph, but it assigns a binary variable to every service-edge pair that can participate in an admissible path. This representation is useful for establishing equivalence with the original constrained model, although the same explicitness becomes progressively expensive when the number of services, time cycles and feasible transitions increases.

The reduced formulation considered here moves part of this structure to a classical preprocessing stage. For each service, complete source-to-destination paths are generated first, and each candidate path already satisfies the source, destination, temporal and processing-state conditions by construction. The binary optimization is then reduced to selecting one path per service under the shared physical resource constraints. This notebook studies that transformation on the routing instance C2 and separates two effects that will remain distinct throughout the paper: loss of feasibility or optimality caused by restricting the candidate sets, and error introduced later by a quantum optimizer.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa import canonical_c2
from oec_qaoa.edge_qubo import build_edge_qubo
from oec_qaoa.ilp import solve_reference_ilp
from oec_qaoa.path_qubo import build_path_qubo, solve_path_qubo_exact
from oec_qaoa.path_reduction import (
    candidate_sets,
    decomposition_gap,
    solve_path_selection,
)

instance = canonical_c2()
reference = solve_reference_ilp(instance)

print(f"Reference objective: {reference.objective:.3f}")

## Routing validation instance C2

C2 adds genuine routing freedom to the first canonical problem. Two services can reach either of two relay satellites, (R_1) and (R_2), before the processing cycle. The first service carries (4) MB before processing and (1) MB afterwards, whereas the second carries (3) MB before processing and (1) MB afterwards.

Relay (R_1) can process at most (4) MB at (t=1), while (R_2) can process at most (3) MB. Both relays have a (4) MB downlink at (t=2). Consequently, (h_0) can be processed only on (R_1), while (h_1) can be processed on either relay. The zero-ground-processing optimum therefore requires the two services to use different relays: (h_0) is routed through (R_1) and (h_1) through (R_2).

Unlike C1, this optimum depends on a routing decision and not only on processing placement. C2 is therefore used as the first regression case for the candidate-path formulation.

In [ ]:
rows = []
all_paths = candidate_sets(instance)

for service_name, paths in all_paths.items():
    for path in paths:
        rows.append(
            {
                "service": service_name,
                "path": path.name,
                "objective": path.objective,
                "edges": " -> ".join(path.edges),
            }
        )

paths_table = pd.DataFrame(rows)
paths_table

Candidate paths are ranked locally by ground-processing objective, path length and a deterministic edge-name tie break. The ranking is intentionally service-local: it does not use information about the choices made by the other services. Shared resource conflicts can therefore appear only when the candidate paths are selected jointly, which is precisely the coupling retained by the reduced optimization problem.

## Effect of the candidate-set size

Let (mathcal{P}_h^{(K)}) denote the first (K) retained candidates for service (h). The nested sequence

[
mathcal{P}_h^{(1)} subseteq mathcal{P}_h^{(2)} subseteq cdots
]

allows the classical reduction to be studied independently of QAOA. A small (K) can remove combinations that are required for joint feasibility, while a larger set can recover the optimum of the direct formulation.

For C2, the best individual route of both services passes through (R_1). Retaining only (K=1) therefore creates a joint conflict at the (R_1) processing resource, even though each candidate is individually feasible. Increasing the set to (K=2) exposes the second on-board route of (h_1) through (R_2), which is sufficient to recover the global optimum.

In [ ]:
records = []

for k in (1, 2):
    paths = candidate_sets(instance, k=k)
    solution = solve_path_selection(instance, paths)

    records.append(
        {
            "K": k,
            "number_of_path_variables": sum(len(v) for v in paths.values()),
            "feasible": solution.success,
            "objective": solution.objective if solution.success else None,
            "decomposition_gap": (
                decomposition_gap(reference.objective, solution)
                if solution.success
                else None
            ),
            "selected_paths": (
                solution.selected_paths if solution.success else None
            ),
        }
    )

pd.DataFrame(records)

This behavior is useful for the interpretation of the decomposition gap. The quantity

[
Delta_K = z_{mathrm{path}}^*(K)-z_{mathrm{edge}}^*
]

is meaningful only when the restricted path model remains feasible. A candidate set can therefore fail in two different ways: it can exclude every jointly feasible combination, or it can remain feasible while increasing the optimum. The numerical study developed later will report feasibility separately from (Delta_K), rather than hiding both effects in a single scalar measure.

## Reduced path-based QUBO

Once the candidate paths have been generated, the semantic binary variable becomes

[
x_{hk} =
egin{cases}
1, & 	ext{if service } h 	ext{ selects path } P_{hk},\\
0, & 	ext{otherwise}.
end{cases}
]

The source, destination and flow-conservation equations no longer appear in the optimization model because they are already satisfied by each complete candidate path. The remaining per-service condition is

[
sum_k x_{hk}=1,
]

while transmission, memory, processing and energy resources continue to couple paths belonging to different services.

The reduction therefore changes the origin of the quantum resource cost. The direct formulation pays for every admissible service-edge decision and its flow penalties; the path formulation pays for the retained complete routes and for the shared-resource constraints that survive preprocessing.

In [ ]:
paths_k2 = candidate_sets(instance, k=2)
path_exact = solve_path_selection(instance, paths_k2)
path_qubo = build_path_qubo(instance, paths_k2)
path_qubo_solution = solve_path_qubo_exact(path_qubo)

resource_comparison = pd.DataFrame(
    [
        {
            "formulation": "direct edge QUBO",
            "semantic_variables": build_edge_qubo(instance).semantic_count,
            "logical_variables": len(build_edge_qubo(instance).variable_names),
            "quadratic_couplings": len(build_edge_qubo(instance).quadratic),
        },
        {
            "formulation": "path QUBO, K=2",
            "semantic_variables": path_qubo.semantic_count,
            "logical_variables": len(path_qubo.variable_names),
            "quadratic_couplings": len(path_qubo.quadratic),
        },
    ]
)
resource_comparison

In [ ]:
print(f"Reduced exact objective: {path_exact.objective:.3f}")
print(f"Path-QUBO energy: {path_qubo_solution.energy:.3f}")
print(f"Exact selected paths: {path_exact.selected_paths}")
print(f"QUBO selected paths: {path_qubo_solution.selected_paths}")

## Equivalence gate for the reduced formulation

At (K=2), the reduced constrained model and the reduced QUBO must agree exactly before the latter can be used in a variational experiment. The same (K=2) model must also recover the direct OEC optimum, which verifies that the chosen candidate set introduces no decomposition error on C2.

In [ ]:
assert path_exact.success
assert path_qubo_solution.success
assert path_exact.objective == reference.objective == 0.0
assert abs(path_qubo_solution.energy - path_exact.objective) < 1e-9
assert path_exact.selected_paths == {
    "h0": "P_h0_0",
    "h1": "P_h1_1",
}
assert path_qubo_solution.selected_paths == {
    "h0": ("P_h0_0",),
    "h1": ("P_h1_1",),
}

print("C2 path-reduction and path-QUBO validation: PASS")

## Interpretation

C2 establishes two properties needed before moving to larger simulations. First, candidate-path preprocessing can reduce the semantic decision space substantially because several edge-level consistency constraints are absorbed into the construction of the paths. Second, the quality of this reduction is controlled by the retained candidate sets, and a locally reasonable truncation can fail globally when several services compete for the same resource.

The next notebook will therefore move away from hand-constructed examples and study how semantic variables, slack variables and quadratic couplings scale with the number of services, time cycles and retained paths. QAOA will be introduced only after this resource analysis has identified instance sizes for which an ideal-state simulation remains both interpretable and computationally manageable.